# ResHeightNet — Colab training (M7)

Run cells top to bottom. The GPU assert in the next cell is deliberately
**first** and runs before any download — free-tier Colab can silently
hand you a CPU-only runtime, and discovering that after a 13.85 GB
download wastes the whole session.

Data lives on `/content` (local disk), never on a Drive mount — Drive
FUSE is far too slow to train from (~9.6 min/epoch of I/O vs ~29s from
local disk), and the 13.85 GB dataset doesn't fit in free Drive's 15 GB
quota alongside checkpoints anyway. Only checkpoints go to Drive.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU available -- change Runtime > Change runtime type > T4 GPU, "
    "or your GPU quota is exhausted. ABORT before downloading 13.85 GB."
)
print(torch.__version__, torch.cuda.get_device_name(0))
assert "T4" in torch.cuda.get_device_name(0), (
    "Not a T4 -- the timing estimates in the execution plan assume a T4; "
    "re-check them on a different GPU before trusting elapsed-time output."
)

In [ ]:
# Do NOT pin torch/torchvision here -- use Colab's preinstalled CUDA
# build. Pinning to 2.6.0 downloads ~2.5GB, can silently land a
# CPU-only wheel, forces a runtime restart, and desyncs torchvision.
!pip install -q h5py huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Clone the repo (replace with your actual remote once pushed).
# The repo carries data/splits/*.txt committed, so this step alone is
# enough to fetch the deterministic split manifests -- no dependency
# on the local DepthWizard checkout.
!git clone https://github.com/<your-username>/resheightnet.git /content/resheightnet
%cd /content/resheightnet

In [ ]:
import os
os.environ["GAMUS_ROOT"] = "/content/gamus"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

!python scripts/fetch_gamus.py --split data/splits/stage_a1_train.txt --split data/splits/stage_a1_val.txt --data-root /content/gamus

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_CKPT_DIR = "/content/drive/MyDrive/resheightnet/checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Write checkpoints to local /content first (fast); train_full already
# does this via --checkpoint-dir. We copy to Drive after each run below
# rather than pointing --checkpoint-dir at Drive directly, since direct
# mid-epoch Drive writes stall.
LOCAL_CKPT_DIR = "/content/ckpt"

In [ ]:
# M6 overfit gate -- run this BEFORE the full training run. If it
# fails, do not proceed to the cell below; see src/train.py's failure
# taxonomy docstring for how to diagnose which symptom you're seeing.
!python -m src.train --overfit 10 --steps 300 --val-split data/splits/stage_a1_val.txt --data-root /content/gamus

In [ ]:
# Full training: 15 epochs, matching DepthWizard's recipe exactly
# (AdamW lr=1e-4 wd=1e-4, CosineAnnealingLR, seed=42, batch=8 @ 512x512).
# Pass --resume-from $LOCAL_CKPT_DIR/last.pt to continue after a
# disconnect (resume correctness is bitwise-verified locally in
# tests/test_integration.py::test_resume_is_bitwise_identical_to_uninterrupted_training).
!python -m src.train \
  --data-root /content/gamus \
  --epochs 15 --batch-size 8 --patch-size 512 \
  --lr 1e-4 --weight-decay 1e-4 --seed 42 \
  --num-workers 2 \
  --checkpoint-dir {LOCAL_CKPT_DIR}

In [ ]:
# Copy checkpoints to Drive so they survive the session.
!cp {LOCAL_CKPT_DIR}/last.pt {LOCAL_CKPT_DIR}/best.pt "{DRIVE_CKPT_DIR}/"

In [ ]:
# M8: evaluate at the DepthWizard-parity protocol (512 center crop,
# pooled per-pixel, unclamped predictions) and write results for
# docs/comparison_table.md.
!python -m src.evaluate \
  --split data/splits/stage_a1_val.txt \
  --data-root /content/gamus \
  --checkpoint {LOCAL_CKPT_DIR}/best.pt \
  --out /content/resheightnet_metrics.json

!cp /content/resheightnet_metrics.json "{DRIVE_CKPT_DIR}/../resheightnet_metrics.json"